# P10 Verification Visualization
Figure text is English only. Korean explanations are provided via print().


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
})

PHASE = "P10"
DATA_DIR = Path("../data/phase10_13")

ver_main = pd.read_csv(DATA_DIR / "verification_main_h3_n20.csv")
ver_support = pd.read_csv(DATA_DIR / "verification_support_p10_n10.csv")
seed_stats = pd.read_csv(DATA_DIR / "verification_setting_seed_stats.csv")
intent = pd.read_csv(DATA_DIR / "intent_vs_observed_summary.csv")
router_family = pd.read_csv(DATA_DIR / "router_family_expert_long.csv")
router_pos = pd.read_csv(DATA_DIR / "router_position_expert_long.csv")

to_num = [
    "n_completed", "best_valid_mrr20", "test_mrr20", "cold_item_mrr20", "long_session_mrr20",
    "diag_top1_max_frac", "diag_cv_usage", "diag_n_eff", "diag_available",
    "best_valid_mean", "best_valid_std", "test_mean", "test_std",
    "cold_mean", "cold_std", "long_session_mean", "long_session_std",
    "family_expert_share_norm", "position_expert_share_norm",
    "delta_best_valid_vs_anchor", "delta_best_test_vs_anchor",
]
for df in [ver_main, ver_support, seed_stats, intent, router_family, router_pos]:
    for col in to_num:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

if PHASE == "P10":
    run_df = ver_support[ver_support["source_phase"] == PHASE].copy()
    seed_df = seed_stats[seed_stats["source_phase"] == PHASE].copy()
    scope_label = "support (H1/H3, n=10)"
else:
    run_df = ver_main[ver_main["source_phase"] == PHASE].copy()
    seed_df = seed_stats[(seed_stats["source_phase"] == PHASE) & (seed_stats["hparam_id"] == "H3")].copy()
    scope_label = "main (H3, n>=20)"

intent_phase = intent[intent["source_phase"] == PHASE].copy()
router_family_phase = router_family[router_family["source_phase"] == PHASE].copy()
router_pos_phase = router_pos[router_pos["source_phase"] == PHASE].copy()

# attach metadata robustly (avoid setting_group suffix collisions)
merge_keys = ["setting_key"]
if "hparam_id" in run_df.columns and "hparam_id" in seed_df.columns:
    merge_keys.append("hparam_id")

group_meta = run_df[merge_keys + ["setting_group"]].drop_duplicates().copy()
seed_df = seed_df.merge(group_meta, on=merge_keys, how="left", suffixes=("", "_from_run"))
if "setting_group_from_run" in seed_df.columns:
    if "setting_group" in seed_df.columns:
        seed_df["setting_group"] = seed_df["setting_group"].fillna(seed_df["setting_group_from_run"])
    else:
        seed_df["setting_group"] = seed_df["setting_group_from_run"]
    seed_df = seed_df.drop(columns=["setting_group_from_run"])

desc_meta = run_df[merge_keys + ["setting_desc", "setting_detail"]].drop_duplicates().copy()
seed_df = seed_df.merge(desc_meta, on=merge_keys, how="left", suffixes=("", "_from_run"))
for col in ["setting_desc", "setting_detail"]:
    from_col = f"{col}_from_run"
    if from_col in seed_df.columns:
        if col in seed_df.columns:
            seed_df[col] = seed_df[col].fillna(seed_df[from_col])
        else:
            seed_df[col] = seed_df[from_col]
        seed_df = seed_df.drop(columns=[from_col])

if PHASE == "P10":
    run_df["setting_plot_key"] = run_df["hparam_id"].astype(str) + "|" + run_df["setting_key"].astype(str)
    seed_df["setting_plot_key"] = seed_df["hparam_id"].astype(str) + "|" + seed_df["setting_key"].astype(str)
else:
    run_df["setting_plot_key"] = run_df["setting_key"].astype(str)
    seed_df["setting_plot_key"] = seed_df["setting_key"].astype(str)

def tight_ylim(ax, values, pad_frac=0.15, min_pad=0.0005):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return
    lo = float(arr.min())
    hi = float(arr.max())
    if np.isclose(lo, hi):
        pad = max(min_pad, abs(lo) * 0.05 + 1e-6)
    else:
        pad = max((hi - lo) * pad_frac, min_pad)
    ax.set_ylim(lo - pad, hi + pad)

def add_bar_labels(ax, fmt="{:.4f}", fontsize=8):
    for p in ax.patches:
        h = p.get_height()
        if not np.isfinite(h):
            continue
        ax.annotate(
            fmt.format(h),
            (p.get_x() + p.get_width() / 2.0, h),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            xytext=(0, 2),
            textcoords="offset points",
        )

print("phase:", PHASE)
print("verification scope:", scope_label)
print("rows (run/seed):", len(run_df), "/", len(seed_df))
print("가설 초점:", "compact feature subset로도 성능을 유지할 수 있는지")


In [ ]:
print("[핵심] setting별 실험 의도(왜 넣었는지 / 무엇이 바뀌었는지)부터 확인합니다.")

if seed_df.empty:
    print("seed table is empty")
else:
    meta_cols = ["setting_plot_key", "setting_group", "setting_desc", "setting_detail"]
    meta = seed_df[meta_cols].drop_duplicates().sort_values(["setting_group", "setting_plot_key"])
    display(meta)

    groups = sorted(meta["setting_group"].dropna().unique().tolist())
    for group in groups:
        m = meta[meta["setting_group"] == group]
        print("\n-", group)
        for _, r in m.iterrows():
            print(f"  * {r['setting_plot_key']}: {r['setting_detail']}")


In [ ]:
print("[핵심] 축 내부 setting별 mean/std 비교입니다 (axis 평균 비교보다 setting 차이에 집중).")

groups = sorted(seed_df["setting_group"].dropna().unique().tolist())
if not groups:
    print("no groups in seed_df")

for group in groups:
    print("\n" + "=" * 90)
    print(f"Group: {group}")
    gs = seed_df[seed_df["setting_group"] == group].copy()
    if gs.empty:
        continue

    show_cols = [
        "setting_plot_key", "setting_desc", "setting_detail",
        "best_valid_mean", "best_valid_std", "test_mean", "test_std",
        "cold_mean", "cold_std", "long_session_mean", "long_session_std",
    ]
    gs = gs.sort_values(["best_valid_mean", "test_mean"], ascending=False)
    display(gs[show_cols].round(6))

    best_valid = gs.sort_values("best_valid_mean", ascending=False).iloc[0]
    best_test = gs.sort_values("test_mean", ascending=False).iloc[0]
    stable = gs.sort_values("best_valid_std", ascending=True).iloc[0]

    print(f"- 추천(valid): {best_valid['setting_plot_key']} ({best_valid['best_valid_mean']:.4f} +/- {best_valid['best_valid_std']:.4f})")
    print(f"- 추천(test) : {best_test['setting_plot_key']} ({best_test['test_mean']:.4f} +/- {best_test['test_std']:.4f})")
    print(f"- 안정성 후보 : {stable['setting_plot_key']} (valid std {stable['best_valid_std']:.4f})")

    # Separate valid/test plots for this group
    x = np.arange(len(gs))
    labels = gs["setting_plot_key"].tolist()

    fig, axes = plt.subplots(1, 2, figsize=(max(12.0, 0.75 * len(gs) + 6.0), 5.0))

    axes[0].bar(
        x,
        gs["best_valid_mean"],
        yerr=gs["best_valid_std"].fillna(0.0),
        capsize=3,
        color="#4C78A8",
        alpha=0.9,
    )
    axes[0].set_title(f"{group}: Mean Best Valid +/- Std")
    axes[0].set_xlabel("Setting")
    axes[0].set_ylabel("Best Valid MRR@20")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=35, ha="right")
    add_bar_labels(axes[0])
    tight_ylim(axes[0], gs["best_valid_mean"] + gs["best_valid_std"].fillna(0.0), pad_frac=0.25, min_pad=0.0004)

    axes[1].bar(
        x,
        gs["test_mean"],
        yerr=gs["test_std"].fillna(0.0),
        capsize=3,
        color="#F58518",
        alpha=0.9,
    )
    axes[1].set_title(f"{group}: Mean Test +/- Std")
    axes[1].set_xlabel("Setting")
    axes[1].set_ylabel("Test MRR@20")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=35, ha="right")
    add_bar_labels(axes[1])
    tight_ylim(axes[1], gs["test_mean"] + gs["test_std"].fillna(0.0), pad_frac=0.25, min_pad=0.0004)

    plt.tight_layout()
    plt.show()


In [ ]:
print("[Seed 분포] setting별 raw seed 분포를 그룹 단위로 확인합니다.")

groups = sorted(run_df["setting_group"].dropna().unique().tolist())
for group in groups:
    print("\n" + "-" * 90)
    print(f"Seed Distribution Group: {group}")

    gr = run_df[run_df["setting_group"] == group].copy()
    if gr.empty:
        continue

    order = (
        seed_df[seed_df["setting_group"] == group]
        .sort_values("best_valid_mean", ascending=False)["setting_plot_key"]
        .tolist()
    )

    if not order:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(max(12.0, 0.70 * len(order) + 6.0), 5.0))

    sns.boxplot(
        data=gr,
        x="setting_plot_key", y="best_valid_mrr20",
        order=order, color="#A0CBE8", showfliers=False, ax=axes[0]
    )
    sns.stripplot(
        data=gr,
        x="setting_plot_key", y="best_valid_mrr20",
        order=order, color="black", alpha=0.65, size=4, jitter=0.12, ax=axes[0]
    )
    axes[0].set_title(f"{group}: Seed Best Valid")
    axes[0].set_xlabel("Setting")
    axes[0].set_ylabel("Best Valid MRR@20")
    axes[0].tick_params(axis="x", rotation=35)
    tight_ylim(axes[0], gr["best_valid_mrr20"], pad_frac=0.25, min_pad=0.0004)

    sns.boxplot(
        data=gr,
        x="setting_plot_key", y="test_mrr20",
        order=order, color="#FFBE7D", showfliers=False, ax=axes[1]
    )
    sns.stripplot(
        data=gr,
        x="setting_plot_key", y="test_mrr20",
        order=order, color="black", alpha=0.65, size=4, jitter=0.12, ax=axes[1]
    )
    axes[1].set_title(f"{group}: Seed Test")
    axes[1].set_xlabel("Setting")
    axes[1].set_ylabel("Test MRR@20")
    axes[1].tick_params(axis="x", rotation=35)
    tight_ylim(axes[1], gr["test_mrr20"], pad_frac=0.25, min_pad=0.0004)

    plt.tight_layout()
    plt.show()


In [ ]:
print("[중요] test/cold/long metric은 분리 플롯으로 봅니다.")

if seed_df.empty:
    print("seed_df is empty")
else:
    specs = [
        ("test_mean", "test_std", "Setting Mean Test", "#F58518"),
        ("cold_mean", "cold_std", "Setting Mean Cold", "#54A24B"),
        ("long_session_mean", "long_session_std", "Setting Mean Long Session", "#B279A2"),
    ]

    for metric, std_col, title, color in specs:
        plot_df = seed_df[["setting_plot_key", metric, std_col]].copy().sort_values(metric, ascending=False)
        if plot_df.empty:
            continue

        x = np.arange(len(plot_df))
        labels = plot_df["setting_plot_key"].tolist()

        plt.figure(figsize=(max(10.0, 0.55 * len(plot_df) + 4.0), 4.8))
        ax = plt.gca()
        ax.bar(
            x,
            plot_df[metric],
            yerr=plot_df[std_col].fillna(0.0),
            capsize=3,
            color=color,
            alpha=0.9,
        )
        ax.set_title(title)
        ax.set_xlabel("Setting")
        ax.set_ylabel("MRR@20")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right")
        for xi, v in zip(x, plot_df[metric]):
            ax.annotate(f"{v:.4f}", (xi, v), ha="center", va="bottom", fontsize=8, xytext=(0, 2), textcoords="offset points")
        tight_ylim(ax, plot_df[metric] + plot_df[std_col].fillna(0.0), pad_frac=0.25, min_pad=0.0004)
        plt.tight_layout()
        plt.show()


In [ ]:
print("[Diag] setting별 router 지표를 봅니다.")

diag = run_df[run_df["diag_available"] == 1].copy()
missing = run_df[run_df["diag_available"] != 1]["run_phase"].tolist()
if missing:
    print("diag missing runs:", missing)

if diag.empty:
    print("diag table empty")
else:
    dsum = (
        diag.groupby(["setting_group", "setting_plot_key"], as_index=False)
            .agg(
                best_valid_mrr20=("best_valid_mrr20", "mean"),
                test_mrr20=("test_mrr20", "mean"),
                diag_top1_max_frac=("diag_top1_max_frac", "mean"),
                diag_cv_usage=("diag_cv_usage", "mean"),
                diag_n_eff=("diag_n_eff", "mean"),
            )
    )

    groups = sorted(dsum["setting_group"].dropna().unique().tolist())
    for group in groups:
        print("\n" + "-" * 90)
        print(f"Diag Group: {group}")
        dg = dsum[dsum["setting_group"] == group].sort_values("best_valid_mrr20", ascending=False)
        display(dg.round(6))

        metrics = [
            ("diag_top1_max_frac", "Top1 Concentration", "#72B7B2"),
            ("diag_cv_usage", "Usage CV", "#E45756"),
            ("diag_n_eff", "Effective Experts", "#4C78A8"),
        ]
        for col, title, color in metrics:
            plt.figure(figsize=(max(8.5, 0.55 * len(dg) + 4.0), 4.2))
            ax = plt.gca()
            ax.bar(dg["setting_plot_key"], dg[col], color=color, alpha=0.9)
            ax.set_title(f"{group}: {title}")
            ax.set_xlabel("Setting")
            ax.set_ylabel(col)
            ax.tick_params(axis="x", rotation=35)
            add_bar_labels(ax)
            tight_ylim(ax, dg[col], pad_frac=0.20, min_pad=0.001)
            plt.tight_layout()
            plt.show()


In [ ]:
print("[Diag Deep] scatter: diag metric vs valid/test를 multi-panel로 확인합니다.")
diag_df = run_df[run_df["diag_available"] == 1].copy()
if diag_df.empty:
    print("diag rows are empty for advanced scatter")
else:
    _ = diag_facet_scatter(
        diag_df,
        target_col="best_valid_mrr20",
        hue_col="setting_group",
        style_col="hparam_id",
        title="Diag vs Valid (P10 Verification)"
    )
    plt.show()
    _ = diag_facet_scatter(
        diag_df,
        target_col="test_mrr20",
        hue_col="setting_group",
        style_col="hparam_id",
        title="Diag vs Test (P10 Verification)"
    )
    plt.show()


In [ ]:
print("[Diag Deep] group-level Spearman correlation heatmap을 확인합니다.")
diag_df = run_df[run_df["diag_available"] == 1].copy()
if diag_df.empty:
    print("diag rows are empty for correlation heatmap")
else:
    _ = diag_group_corr_heatmap(
        diag_df,
        group_col="setting_group",
        target_col="best_valid_mrr20",
        title="Group-level Spearman Corr (Diag vs Valid, P10 Verification)"
    )
    plt.show()
    _ = diag_group_corr_heatmap(
        diag_df,
        group_col="setting_group",
        target_col="test_mrr20",
        title="Group-level Spearman Corr (Diag vs Test, P10 Verification)"
    )
    plt.show()


In [ ]:
print("[Diag Deep] diag quantile trend를 확인합니다.")
diag_df = run_df[run_df["diag_available"] == 1].copy()
if diag_df.empty:
    print("diag rows are empty for quantile trend")
else:
    _ = diag_quantile_profile(
        diag_df,
        target_col="best_valid_mrr20",
        title="Diag Quantile Trend (Valid, P10 Verification)"
    )
    plt.show()
    _ = diag_quantile_profile(
        diag_df,
        target_col="test_mrr20",
        title="Diag Quantile Trend (Test, P10 Verification)"
    )
    plt.show()


In [ ]:
print("[Router Deep] family-expert PCA scatter를 확인합니다.")
if seed_df.empty or router_family_phase.empty:
    print("router family rows are empty for PCA")
else:
    top_plot_keys = seed_df.sort_values("best_valid_mean", ascending=False)["setting_plot_key"].head(10).tolist()
    key_map = run_df[["setting_key", "hparam_id", "setting_plot_key"]].drop_duplicates()
    pca_df = router_family_phase.merge(key_map, on=["setting_key", "hparam_id"], how="inner")
    pca_df = pca_df[pca_df["setting_plot_key"].isin(top_plot_keys)].copy()
    if pca_df.empty:
        print("router family rows empty after top-key filter for PCA")
    else:
        _ = family_expert_pca_scatter(pca_df, title="Feature-Family PCA (P10 Verification)")
        plt.show()


In [ ]:
print("[Router-Family] 상위 setting 중심 family usage를 확인합니다.")

if seed_df.empty or router_family_phase.empty:
    print("router family empty")
else:
    top_plot_keys = seed_df.sort_values("best_valid_mean", ascending=False)["setting_plot_key"].head(8).tolist()

    key_map = run_df[["setting_key", "hparam_id", "setting_plot_key"]].drop_duplicates()
    rf = router_family_phase.merge(key_map, on=["setting_key", "hparam_id"], how="inner")
    rf = rf[rf["setting_plot_key"].isin(top_plot_keys)].copy()

    if rf.empty:
        print("router family rows empty after merge/filter")
    else:
        stage_candidates = rf["stage_name"].dropna()
        if stage_candidates.empty:
            print("stage_name is empty after merge/filter")
        else:
            stage_mode = stage_candidates.mode().iloc[0]
            rf = rf[rf["stage_name"] == stage_mode].copy()

            fam = (
                rf.groupby(["setting_plot_key", "family"], as_index=False)["family_expert_share_norm"]
                  .mean()
            )
            heat = fam.pivot(index="setting_plot_key", columns="family", values="family_expert_share_norm").fillna(0.0)
            order = [k for k in top_plot_keys if k in heat.index]
            heat = heat.reindex(order)

            h = max(3.4, 0.45 * len(heat.index) + 1.5)
            plt.figure(figsize=(8.8, h))
            sns.heatmap(
                heat,
                annot=True,
                fmt=".2f",
                cmap="YlGnBu",
                annot_kws={"size": 8},
                cbar_kws={"label": "Mean Share"},
            )
            plt.title("Family Usage Heatmap")
            plt.xlabel("Family")
            plt.ylabel("Setting")
            plt.tight_layout()
            plt.show()
            print("selected stage:", stage_mode)


In [ ]:
print("[Router-Position] 상위 setting 중심 position concentration을 확인합니다.")

if seed_df.empty or router_pos_phase.empty:
    print("router position empty")
else:
    top_plot_keys = seed_df.sort_values("best_valid_mean", ascending=False)["setting_plot_key"].head(8).tolist()

    key_map = run_df[["setting_key", "hparam_id", "setting_plot_key"]].drop_duplicates()
    rp = router_pos_phase.merge(key_map, on=["setting_key", "hparam_id"], how="inner")
    rp = rp[rp["setting_plot_key"].isin(top_plot_keys)].copy()

    if rp.empty:
        print("router position rows empty after merge/filter")
    else:
        stage_candidates = rp["stage_name"].dropna()
        if stage_candidates.empty:
            print("stage_name is empty after merge/filter")
        else:
            stage_mode = stage_candidates.mode().iloc[0]
            rp = rp[rp["stage_name"] == stage_mode].copy()

            pos_top = (
                rp.groupby(["setting_plot_key", "position_index"], as_index=False)["position_expert_share_norm"]
                  .max()
            )
            heat = pos_top.pivot(index="setting_plot_key", columns="position_index", values="position_expert_share_norm").fillna(0.0)

            cols = pd.to_numeric(pd.Index(heat.columns), errors="coerce")
            order_idx = np.argsort(np.where(np.isnan(cols), 1e9, cols))
            heat = heat.iloc[:, order_idx]
            if heat.shape[1] > 20:
                heat = heat.iloc[:, :20]

            order = [k for k in top_plot_keys if k in heat.index]
            heat = heat.reindex(order)

            w = min(16.0, 0.55 * heat.shape[1] + 4.5)
            h = max(3.4, 0.45 * len(heat.index) + 1.5)
            plt.figure(figsize=(w, h))
            sns.heatmap(
                heat,
                annot=False,
                cmap="magma",
                cbar_kws={"label": "Max Expert Share"},
            )
            plt.title("Position-wise Concentration Heatmap")
            plt.xlabel("Position Index")
            plt.ylabel("Setting")
            plt.tight_layout()
            plt.show()
            print("selected stage:", stage_mode)


In [ ]:
print("[가설 대비] verification 관찰과 주장 문장을 정리합니다.")

if intent_phase.empty:
    print("intent table is empty")
else:
    cols = [
        "setting_group", "plan_intent", "expected_pattern", "observed_tag",
        "best_setting_key", "delta_best_valid_vs_anchor", "delta_best_test_vs_anchor", "match_flag"
    ]
    display(intent_phase[cols].sort_values("delta_best_valid_vs_anchor", ascending=False))

if not seed_df.empty:
    best_valid = seed_df.sort_values("best_valid_mean", ascending=False).iloc[0]
    best_test = seed_df.sort_values("test_mean", ascending=False).iloc[0]
    stable = seed_df.sort_values("best_valid_std", ascending=True).iloc[0]

    print("권장 서술 초안:")
    print(
        "- " + PHASE
        + " verification에서는 " + "compact feature subset로도 성능을 유지할 수 있는지"
        + " 가설을 setting별 seed 통계로 점검했다. "
        + "valid 대표는 " + str(best_valid["setting_plot_key"]) + f" ({best_valid['best_valid_mean']:.4f} +/- {best_valid['best_valid_std']:.4f}), "
        + "test 대표는 " + str(best_test["setting_plot_key"]) + f" ({best_test['test_mean']:.4f} +/- {best_test['test_std']:.4f}), "
        + "안정성 후보는 " + str(stable["setting_plot_key"]) + f" (valid std {stable['best_valid_std']:.4f})였다."
    )
